# Create Dataset for Card Strength Training

This notebook orchestrates the creation of a comprehensive dataset for training card strength models. It combines Scryfall card data with EDHREC popularity metrics to create target labels.

**Workflow:**
- Load and filter raw Scryfall data for Commander-legal cards.
- Normalize and clean card text for transformer inputs.
- Fetch and join EDHREC inclusion data (popularity) to serve as training labels.
- Generate text embeddings using pre-trained sentence transformers.
- Engineer numerical and boolean features from card attributes.
- Export the final dataset to Parquet for model training.

### 1. Environment Setup

- Import necessary libraries for data processing and machine learning.
- Load custom project modules for filtering and cleaning.

In [1]:
import os
import pandas as pd
import numpy as np
import json
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sentence_transformers import SentenceTransformer

from edhrec_card_scraper import sluggify_card_name
from src.dataset_filter import filter_dataset


### 2. Configuration

- Define input and output file paths.
- Set directory for EDHREC card data.

In [ ]:
INPUT_DATA_PATH = "data/scryfall/oracle-cards-20251022090300.json"
OUTPUT_DATA_PATH = "card_strength_dataset.parquet"
EDHREC_CARD_DIRECTORY = "data/edhrec/cards"

### 3. Data Loading and Filtering

- Read raw Scryfall JSON data.
- Apply `filter_dataset` to keep only Commander-legal cards and clean encodings.

In [3]:
df = pd.read_json(INPUT_DATA_PATH)
df = filter_dataset(df)

print(f"shape: {df.shape}")
df.head()

### 4. Handle Multi-face Cards

- Define `prepare_transformer_input` to flatten multi-face cards into a single text string.
- Clean reminder text and normalize card name references to 'CARD_NAME'.
- Apply the transformation to create a `transformer_input` column.

In [4]:
def prepare_transformer_input(row):
    faces_text = []
    
    # 1. Check if it's a multi-face card (card_faces exists and is a list)
    card_faces = row.get('card_faces')
    
    if isinstance(card_faces, list) and len(card_faces) > 0:
        # Loop through each face
        for face in card_faces:
            face_name = str(face.get('name', ''))
            face_type = str(face.get('type_line', ''))
            face_oracle = str(face.get('oracle_text', ''))
            
            # Clean reminder text
            cleaned_oracle = re.sub(r'\(.*?\)', '', face_oracle)
            
            # Replace the specific face's name
            if face_name:
                cleaned_oracle = cleaned_oracle.replace(face_name, "CARD_NAME")
                
            cleaned_oracle = re.sub(r'\s+', ' ', cleaned_oracle).strip()
            
            faces_text.append(f"Type: {face_type}. Effect: {cleaned_oracle}")
            
        final_string = " // ".join(faces_text)
        
    # 2. Handle normal, single-face cards
    else:
        name = str(row['name'])
        type_line = str(row['type_line'])
        oracle_text = str(row['oracle_text']) if pd.notna(row['oracle_text']) else ""
        
        # Clean reminder text
        cleaned_oracle = re.sub(r'\(.*?\)', '', oracle_text)
        
        # Replace the card's name
        cleaned_oracle = cleaned_oracle.replace(name, "CARD_NAME")
        cleaned_oracle = re.sub(r'\s+', ' ', cleaned_oracle).strip()
        
        final_string = f"Type: {type_line}. Effect: {cleaned_oracle}"
        
    return final_string

df['transformer_input'] = df.apply(prepare_transformer_input, axis=1)

### 5. Join EDHREC Inclusion Data

- Iterate through cards and match them with scraped EDHREC JSON files.
- Calculate `inclusion_rate` and `log_inclusion_rate` (Popularity / Potential Decks).
- Drop basic lands and cards without available popularity data.

In [5]:
# Lists to store our new data and the rows we need to delete
new_data = []
indices_to_drop = []

for idx, row in df.iterrows():
    name = row["name"]

    if name in ["Forest", "Mountain", "Swamp", "Island", "Plains"]:
        indices_to_drop.append(idx)
        continue

    slugs = sluggify_card_name(name)
    if isinstance(slugs, str):
        slugs = [slugs]

    inclusion = potential_decks = inclusion_rate = log_inclusion_rate = 0
    for slug in slugs:
        filename = f"{slug}.json"
        path = os.path.join(EDHREC_CARD_DIRECTORY, filename)
        if not os.path.exists(path):
            continue
        try:
            with open(path, 'r', encoding='utf-8') as f:
                json_dict = json.load(f)
            inclusion = json_dict["container"]["json_dict"]["card"]["inclusion"]
            potential_decks = json_dict["container"]["json_dict"]["card"]["potential_decks"]
            inclusion_rate = np.float64(inclusion) / np.float64(potential_decks)
            log_inclusion_rate = np.log10(inclusion_rate)
            break
        except (KeyError, json.JSONDecodeError):
            continue

    if inclusion == 0:
        indices_to_drop.append(idx)
        continue

    new_data.append({
        'index': idx,
        'inclusion': inclusion,
        'potential_decks': potential_decks,
        'inclusion_rate': inclusion_rate,
        'log_inclusion_rate': log_inclusion_rate
    })

df = df.drop(index=indices_to_drop)
results_df = pd.DataFrame(new_data).set_index('index')
df = df.join(results_df)

print(f"Processing complete. {len(indices_to_drop)} rows dropped. Current DataFrame shape: {df.shape}")

### 6. Create Power Scores

- Scale log-inclusion rates to a standardized 25-99 range to create `log10_power_score`.
- Visualize the score distribution to ensure data quality.

In [7]:
MIN_SCORE = 25
MAX_SCORE = 99
log_min = df['log_inclusion_rate'].min()
log_max = df['log_inclusion_rate'].max()
df['log10_power_score'] = MIN_SCORE + ((df['log_inclusion_rate'] - log_min) * (MAX_SCORE - MIN_SCORE)) / (log_max - log_min)

print(f"Scores calculated! Range: {df['log10_power_score'].min():.4f} to {df['log10_power_score'].max():.4f}")
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))
sns.histplot(df['log10_power_score'], bins=70, kde=True, color='indigo')
plt.title('Distribution of MTG Card Power Scores (25-99)', fontsize=16, fontweight='bold')
plt.show()

### 7. Generate Text Embeddings

- Load the `all-MiniLM-L6-v2` Sentence Transformer model.
- Encode the card text into 384-dimensional dense vectors.

In [8]:
print("Loading all-MiniLM-L6-v2...")
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['transformer_input'].tolist(), batch_size=256, show_progress_bar=True, convert_to_numpy=True)
print(f"Text Embeddings generated! Shape: {embeddings.shape}")

### 8. Feature Engineering

- **Numerical:** Extract CMC, Power, Toughness, Loyalty, and Color count.
- **Boolean:** Flag specific card types (Planeswalker, Legendary, Saga) and attributes (X-cost, Modal, Transform).
- Concatenate embeddings with engineered features into a final matrix `X`.

In [ ]:
def safe_to_float(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace('*', '0').replace('+', '')
    try:
        return float(val)
    except ValueError:
        return np.nan
    
def count_colors(val):
    if isinstance(val, list):
        return len(val)
    if isinstance(val, str):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return len(parsed)
        except (ValueError, SyntaxError):
            pass
    return 0

def check_is_modal(row):
    if '•' in str(row.get('oracle_text', '')):
        return 1
    faces = row.get('card_faces')
    if isinstance(faces, list) and len(faces) > 1:
        all_spells = all(('Instant' in str(face.get('type_line', '')) or 'Sorcery' in str(face.get('type_line', ''))) for face in faces)
        if all_spells:
            return 1
    return 0

def check_is_transform(row):
    faces = row.get('card_faces')
    if isinstance(faces, list) and len(faces) > 1:
        all_permanents = all(('Instant' not in str(face.get('type_line', '')) and 'Sorcery' not in str(face.get('type_line', ''))) for face in faces)
        if all_permanents:
            return 1
    return 0

df['num_cmc'] = pd.to_numeric(df['cmc'], errors='coerce').fillna(0)
df['num_power'] = df['power'].apply(safe_to_float)
df['num_toughness'] = df['toughness'].apply(safe_to_float)
df['num_loyalty'] = df['loyalty'].apply(safe_to_float)
df['num_colors'] = df['color_identity'].apply(count_colors)

df['bool_is_game_changer'] = df['game_changer'].fillna(False).astype(int)
df['bool_power_var'] = df['power'].astype(str).str.contains(r'[\*X]', na=False).astype(int)
df['bool_toughness_var'] = df['toughness'].astype(str).str.contains(r'[\*X]', na=False).astype(int)
df['bool_is_planeswalker'] = df['type_line'].astype(str).str.contains('Planeswalker', case=False, na=False).astype(int)
df['bool_has_x_cost'] = df['mana_cost'].astype(str).str.contains('{X}', case=False, na=False).astype(int)
df['bool_is_legendary'] = df['type_line'].astype(str).str.contains('Legendary', case=False, na=False).astype(int)
df['bool_is_colorless'] = df['color_identity'].apply(lambda x: 1 if (isinstance(x, list) and len(x) == 0) or str(x) == '[]' else 0)
df['bool_is_modal'] = df.apply(check_is_modal, axis=1)
df['bool_is_transform'] = df.apply(check_is_transform, axis=1)
df['bool_is_saga'] = df['type_line'].astype(str).str.contains(r'\bSaga\b', case=False, na=False).astype(int)
df['bool_is_class'] = df['type_line'].astype(str).str.contains(r'\bClass\b', case=False, na=False).astype(int)

boolean_cols = [col for col in df.columns if col.startswith('bool_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
X = np.hstack((embeddings, df[numerical_cols].fillna(-999).values, df[boolean_cols].values))
y = df['log10_power_score'].values

### 9. Dataset Export

- Concatenate all engineered features into a single final DataFrame.
- Save the resulting dataset as a Parquet file for high-performance retrieval during training.

In [30]:
print("Attaching embeddings to the DataFrame...")
embed_cols = [f"embed_{i}" for i in range(embeddings.shape[1])]
embeddings_df = pd.DataFrame(embeddings, columns=embed_cols, index=df.index)
df_final = pd.concat([df, embeddings_df], axis=1)
print(f"Exporting to {OUTPUT_DATA_PATH}...")
df_final.to_parquet(OUTPUT_DATA_PATH, index=False)
print("Export complete!")